In [1]:
import sys
from typing import List
import requests
import time
import json
import pytz
from pydantic import BaseModel, model_serializer
from requests_cache import install_cache
sys.path.append('/Users/robertcampbell/sqlalchemy-tutorial/')
from ptypes.player import PlayerStatistic, Dribbles
from ptypes.responses import PlayerResponse
from schema.schema import Fixture, Statistic, Team, Player, Venue

In [2]:
from main import session

In [3]:
install_cache('http_cache', backend='sqlite', expire_after=86400)

In [4]:
SPORTS_API_KEY = 'c59db8444176b9d9b860c1844640cfca'
BASE_URL = "https://v3.football.api-sports.io"

In [7]:
def get_data(endpoint, params={}):
    ''' get league data '''
    headers = {
        'x-rapidapi-host': "v3.football.api-sports.io",
        'x-rapidapi-key': SPORTS_API_KEY
    }

    response = requests.get(f"{BASE_URL}/{endpoint}", headers=headers, params=params)
    data = response.json()
    return data

In [4]:
response = get_data("teams", {"league": 39, "season": 2021})

In [5]:
response

{'get': 'teams',
 'parameters': {'league': '39', 'season': '2021'},
 'errors': [],
 'results': 20,
 'paging': {'current': 1, 'total': 1},
 'response': [{'team': {'id': 33,
    'name': 'Manchester United',
    'code': 'MUN',
    'country': 'England',
    'founded': 1878,
    'national': False,
    'logo': 'https://media.api-sports.io/football/teams/33.png'},
   'venue': {'id': 556,
    'name': 'Old Trafford',
    'address': 'Sir Matt Busby Way',
    'city': 'Manchester',
    'capacity': 76212,
    'surface': 'grass',
    'image': 'https://media.api-sports.io/football/venues/556.png'}},
  {'team': {'id': 34,
    'name': 'Newcastle',
    'code': 'NEW',
    'country': 'England',
    'founded': 1892,
    'national': False,
    'logo': 'https://media.api-sports.io/football/teams/34.png'},
   'venue': {'id': 562,
    'name': "St. James' Park",
    'address': 'St. James&apos; Street',
    'city': 'Newcastle upon Tyne',
    'capacity': 52758,
    'surface': 'grass',
    'image': 'https://media.

In [5]:
for team in response['response']:
    v = Venue(**team['venue'], team_id=team['team']['id'])
    session.add(v)
session.commit()

In [8]:
teams: List[Team] = Team.query.limit(44).all()

In [9]:
venues = set([v.id for v in Venue.query.all()])

In [15]:
len(teams)

44

In [10]:
for team in teams:
    if not team.venue:
        res = get_data("teams", {"id": team.id})
        # time.sleep(6) # avoid rate limits
        try:
            v = Venue(**res['response'][0]['venue'], team_id=team.id)
            if v.id not in venues:
                session.add(v)
        except Exception as error:
            print(f"{team.id}: \n\n ERROR: {error}")
            continue

In [11]:
session.commit()

In [26]:
res = get_data("teams", {"id": teams[60].id})

In [27]:
res['response'][0]

{'team': {'id': 94,
  'name': 'Rennes',
  'code': 'REN',
  'country': 'France',
  'founded': 1901,
  'national': False,
  'logo': 'https://media.api-sports.io/football/teams/94.png'},
 'venue': {'id': 680,
  'name': 'Roazhon Park',
  'address': '111, route de Lorient',
  'city': 'Rennes',
  'capacity': 31127,
  'surface': 'grass',
  'image': 'https://media.api-sports.io/football/venues/680.png'}}

In [14]:
teams[0].id

33

In [13]:
len(Venue.query.all())

44